Hydromt Test Rio

In [1]:
# import required packages
import os
from hydromt_fiat.fiat import FiatModel
from hydromt.log import setuplog
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import shutil

%matplotlib inline

# Pre-processing Assets and Occupancy

### Example functions for Hydromt Fiat

In [ ]:
# Starting point are building footprints
##Assets - simple gpd without any data
def pre_process_assets(
        building_footprints_fn: Union[str, Path],
        output_fn: Union[str, Path]):
    assets = gpd.read(building_footprints_fn)
    column_to_keep = 'geometry'
    assets = assets[column_to_keep]
    gdf.to_file(output_fn)
    return assets, output_fn

## Occupancy  - when BF data and occupancy data are seperately available as spatial data
def pre_process_occupancy(building_footprints_fn: Union[str, Path],
                          occupancy_fn: Union[str, Path],
                          crs: str,
                          occupancy_attr: Union[str, list],
                          dic_occupancy: dict,
                          output_fn: Union[str, Path],
                          fill_value: str = 'residential',
                          common_column: str = None):
    bf = gpd.read_file(building_footprints_fn)
    
    # If occupancy in form of csv file - merge on common column. If occupancy in form of spatial data - spatial join
    if occupancy_fn.endswith('.csv') or occupancy_fn.endswith('.xslx'):
        occupancy_entrances = pd.read_csv(occupancy_fn)
        df = bf.merge(occupancy_entrances, on=common_column, how='left')
        gdf = gpd.GeoDataFrame(df)
    else:
        occupancy_entrances = gpd.read_file(occupancy_fn)
        if occupancy_entrances.crs != bf.crs:
            crs = occupancy_entrances.crs
            bf = bf.to_crs(crs)
            gdf = gpd.sjoin(bf, occupancy_entrances)
            gdf = gdf[["geometry", occupancy_attr]]
            #gdf.to_file(output_fn)
    
    # Translate to JRC
    if isinstance(occupancy_attr, str):
        gdf[ "Primary Object Type"] = gdf[occupancy_attr].map(dic_occupancy).fillna(fill_value)
        del occupancy[occupancy_attr]
    elif isinstance(occupancy_attr, list):
        gdf[ "Primary Object Type"] = gdf[occupancy_attr[0]].map(dic_occupancy["Primary Object Type"]).fillna(fill_value)
        gdf[ "Secondary Object Type"] = gdf[occupancy_attr[1]].map(dic_occupancy["Secondary Object Type"]).fillna(fill_value)
    gdf.to_file(output_fn)
    
    return gdf, output_fn


 # Dict example Translation to JRC Curves:
 ## dic_occupancy_to_jrc_damages = {'Commercial / Service': 'commercial' ,'Residential': 'residential','Temples, Churches, etc.': 'commercial', 'Mixed (commercial and residential)': 'commercial', 'Industrial': 'industrial',  'Public': 'commercial', 'Empty Land': 'residential','Abandoned': 'residential', 'Others': 'residential'}   

## Create data catalog entries
def catalog_entry_assets(asset_fn: Union[str, Path] = None, crs: str = 4326):
    if asset_fn is None:
        _, output = pre_process_assets(building_footprints_fn, output_fn)
    make_catalog_entry(
                name="assets",
                path=output,
                data_type=DataType.GeoDataFrame,
                driver=Driver.vector,
                crs=crs,
                meta={"category": Category.exposure},
            )
def catalog_entry_occupancy(occupancy_fn: Union[str, Path] = None, crs: str = 4326, 
                        building_footprints_fn: Union[str, Path]  = None,
                          occupancy_attr: Union[str, list]  = None,
                          dic_occupancy: dict  = None,
                          output_fn: Union[str, Path] = None):
    if occupancy_fn is None:
        _, output = pre_process_occupancy(building_footprints_fn, occupancy_fn, crs, occupancy_attr, dic_occupancy,output_fn)
    
    else:
        make_catalog_entry(
                name="occupancy",
                path=output,
                data_type=DataType.GeoDataFrame,
                driver=Driver.vector,
                crs=crs,
                meta={"category": Category.exposure},
            )


### Manual pre-processing

In [51]:
# pre process occupancy
## Spatial joint BF and entrances to get cod_uso
bf = gpd.read_file(r'C:\Users\rautenba\OneDrive - Stichting Deltares\Documents\Projects\UP_2030_Rio_de_janeiro\pre-processed_data\building_footprints.gpkg')
occupancy_entrances = gpd.read_file(r'C:\Users\rautenba\OneDrive - Stichting Deltares\Documents\Projects\UP_2030_Rio_de_janeiro\pre-processed_data\entrances.gpkg')
crs = occupancy_entrances.crs
bf = bf.to_crs(crs)
gdf = gpd.sjoin(bf, occupancy_entrances)
gdf = gdf[["geometry", "cod_uso"]]
gdf.to_file(r'C:\Users\rautenba\OneDrive - Stichting Deltares\Documents\Projects\UP_2030_Rio_de_janeiro\pre-processed_data\occupancy_preprocessed_SR.gpkg')

In [52]:
# pre process occupancy
occupancy = gpd.read_file(r'C:\Users\rautenba\OneDrive - Stichting Deltares\Documents\Projects\UP_2030_Rio_de_janeiro\pre-processed_data\occupancy_preprocessed_SR.gpkg')
dic_occupancy = {1: 'Commercial / Service', 2: 'Residential', 3: 'Temples, Churches, etc.', 4: 'Mixed (commercial and residential)', 5: 'Industrial' , 6 : 'Public',7 : 'Empty Land', 8: 'Abandoned', 9: 'Others'}
occupancy[ "Primary Object Type"] = occupancy['cod_uso'].map(dic_occupancy).fillna('residential')
del occupancy['cod_uso']
occupancy.to_file(r'C:\Users\rautenba\OneDrive - Stichting Deltares\Documents\Projects\UP_2030_Rio_de_janeiro\pre-processed_data\occupancy_preprocessed_mapped_SR.gpkg')

In [53]:
# Translate occupancy to JRC Damage Curves
dic_occupancy_to_jrc_damages = {'Commercial / Service': 'commercial' ,'Residential': 'residential','Temples, Churches, etc.': 'commercial', 'Mixed (commercial and residential)': 'commercial', 'Industrial': 'industrial',  'Public': 'commercial', 'Empty Land': 'residential','Abandoned': 'residential', 'Others': 'residential'}
occupancy[ "Primary Object Type"] = occupancy["Primary Object Type"].map(dic_occupancy_to_jrc_damages).fillna('residential')
occupancy.to_file(r'C:\Users\rautenba\OneDrive - Stichting Deltares\Documents\Projects\UP_2030_Rio_de_janeiro\pre-processed_data\occupancy_object_types_mapped_SR.gpkg')

# Create model

In [2]:
model_root = r"C:\Users\rautenba\OneDrive - Stichting Deltares\Documents\Projects\UP_2030_Rio_de_janeiro\FIAT_model"  
model_name = "Rio de Janeiro"  

In [3]:
model_folder = Path(model_root) / model_name 
data_catalog = (
    Path(os.path.abspath("")) / "data" / "hydromt_fiat_catalog_global.yml"
)   ## Create a data catalog (?)
 
logger_name = "hydromt_fiat"  
logger = setuplog(logger_name, log_level=10)  

2024-10-07 14:59:23,896 - hydromt_fiat - log - INFO - HydroMT version: 0.9.4


In [4]:
## Get studie region
region = gpd.read_file(r"C:\Users\rautenba\OneDrive - Stichting Deltares\Documents\Projects\UP_2030_Rio_de_janeiro\region.gpkg")
continent = "South America"
country = "Brazil"

In [5]:
### Setup vulnerability parameters ###
vulnerability_fn = "jrc_vulnerability_curves"    # Curves specified in datacatalog!
vulnerability_identifiers_and_linking_fn = "jrc_vulnerability_curves_linking"  # Curves specified in datacatalog!
unit = "m"

### Setup exposure parameters ###
asset_locations = "asset_location"    # give different name  - new functinality?
occupancy_type = "occupancy"     # give different name  - new functinality?
keep_unclassified = False
max_potential_damage = "jrc_damage_values"    #Damages specified in datacatalog!
ground_floor_height = 0 # data?
damage_types = ["total"]  
unit = "m"

### Setup output parameters ###
output_dir = "output"
output_csv_name = "output.csv"
output_vector_name = "spatial.gpkg"

In [6]:
configuration = {
    "setup_output": {
        "output_dir": output_dir,
        "output_csv_name": output_csv_name,
        "output_vector_name": output_vector_name,
    },
    "setup_vulnerability": {
        "vulnerability_fn": vulnerability_fn,
        "vulnerability_identifiers_and_linking_fn": vulnerability_identifiers_and_linking_fn,
        "continent": continent,
        "unit": unit,
    },
    "setup_exposure_buildings": {
        "asset_locations": asset_locations,
        "occupancy_type": occupancy_type,
        "keep_unclassified": keep_unclassified,
        "max_potential_damage": max_potential_damage,
        "ground_floor_height": ground_floor_height,
        "unit": unit,
        "damage_types": damage_types,
        "country": country,
    },
}

In [ ]:
if model_folder.exists():
    shutil.rmtree(model_folder)
fiat_model = FiatModel(
    root=model_folder, mode="w", data_libs=[data_catalog], logger=logger
)

In [7]:
if model_folder.exists():
    shutil.rmtree(model_folder)
fiat_model = FiatModel(
    root=model_folder, mode="w", data_libs=[data_catalog], logger=logger
)

2024-10-07 14:59:30,571 - hydromt_fiat - data_catalog - INFO - Parsing data catalog from c:\Users\rautenba\repos\hydromt_fiat\examples\data\hydromt_fiat_catalog_global.yml
2024-10-07 14:59:30,578 - hydromt_fiat - data_catalog - INFO - Parsing data catalog from C:\Users\rautenba\repos\hydromt_fiat\hydromt_fiat\data\hydromt_fiat_catalog_global.yml
2024-10-07 14:59:30,590 - hydromt_fiat - log - DEBUG - Writing log messages to new file C:\Users\rautenba\OneDrive - Stichting Deltares\Documents\Projects\UP_2030_Rio_de_janeiro\FIAT_model\Rio de Janeiro\hydromt.log.
2024-10-07 14:59:30,591 - hydromt_fiat - model_api - INFO - Initializing fiat model from hydromt_fiat (v0.3.2.dev0).


In [8]:
fiat_model.build(region={"geom": region}, opt=configuration, write=False)

2024-10-07 14:59:31,910 - hydromt_fiat - model_api - INFO - setup_region.region: {'geom':                                             geometry
0  POLYGON ((656230.254 7478196.68, 677573.561 74...}
2024-10-07 14:59:31,913 - hydromt_fiat - basin_mask - DEBUG - Parsed region (kind=geom): {'geom': 'GeoDataFrame [ 656230.25352517 7464397.39926658  677573.56088077 7478416.71455465] (crs = EPSG:31983)'}
2024-10-07 14:59:31,920 - hydromt_fiat - model_api - INFO - setup_output.output_dir: output
2024-10-07 14:59:31,922 - hydromt_fiat - model_api - INFO - setup_output.output_csv_name: output.csv
2024-10-07 14:59:31,922 - hydromt_fiat - model_api - INFO - setup_output.output_vector_name: spatial.gpkg
2024-10-07 14:59:31,924 - hydromt_fiat - model_api - ERROR - Default config file not found at C:\Users\rautenba\repos\hydromt_fiat\hydromt_fiat\data\fiat\settings.toml
2024-10-07 14:59:31,925 - hydromt_fiat - model_api - INFO - setup_vulnerability.vulnerability_fn: jrc_vulnerability_curves
2024-10-07

In [9]:
fiat_model.write()

2024-10-07 15:00:40,074 - hydromt_fiat - fiat - INFO - Updating all data objects...
2024-10-07 15:00:40,080 - hydromt_fiat - model_api - WARNING - Replacing geom: region
2024-10-07 15:00:40,081 - hydromt_fiat - fiat - INFO - Writing model data to C:\Users\rautenba\OneDrive - Stichting Deltares\Documents\Projects\UP_2030_Rio_de_janeiro\FIAT_model\Rio de Janeiro
2024-10-07 15:00:41,874 - hydromt_fiat - model_api - DEBUG - Writing file geoms/region.geojson
2024-10-07 15:00:41,884 - hydromt_fiat - fiat - INFO - Writing model exposure table file to exposure/exposure.csv.
2024-10-07 15:00:42,311 - hydromt_fiat - fiat - INFO - Writing model vulnerability_curves table file to vulnerability/vulnerability_curves.csv.
2024-10-07 15:00:42,315 - hydromt_fiat - fiat - INFO - Writing model vulnerability_identifiers table file to vulnerability/vulnerability_identifiers.csv.
2024-10-07 15:00:42,318 - hydromt_fiat - model_api - INFO - Writing model config to C:\Users\rautenba\OneDrive - Stichting Deltar